In [ ]:
# CELL 1: Setup, Environment Switch, and Path Configuration
import os
import sys
from pathlib import Path

# ============================================================================
# --- ENVIRONMENT SWITCH ---
# Set to True if running in Google Colab.
# Set to False if running locally in Windows PC VSCode.
# ============================================================================
RUNNING_IN_COLAB = True

if RUNNING_IN_COLAB:
    from google.colab import drive

    # 1. Mount Google Drive
    drive.mount("/content/drive")

    # 2. Define codebase paths
    project_path = Path("/content/drive/MyDrive/RLVR_Project")
    codebase_path = project_path / "notebooks_RLVR_v2"
else:
    # 1. Local Windows VSCode Paths
    # Adjust these paths if your local directory layout differs
    project_path = Path("C:/Users/ping/Files_win10/python/py311/stocks")
    codebase_path = project_path / "notebooks_RLVR_v2"

# 3. Change the Current Working Directory
os.chdir(codebase_path)

# 4. Add to system path so absolute imports work
if str(codebase_path) not in sys.path:
    sys.path.append(str(codebase_path))

# ==========================================================
# [GUARD] DEBUG TRAP: Verify environment visibility
# ==========================================================
print("\n--- Environment Check ---")
print(f"Current Directory: {os.getcwd()}")

visible_files = os.listdir()
print(f"Files visible here: {visible_files}")

if "core" in visible_files and "rl_discovery" in visible_files:
    print("\n[OK] SUCCESS: Python can see your architecture! Imports will now work.")
else:
    print("\n[ERROR] FAILURE: Python cannot see 'core' or 'rl_discovery'.")
    print("Please check your directory configuration and folder structure.")

In [ ]:
# CELL 2: Load Data
import pandas as pd

print("Loading DataFrames into RAM. Please wait...")
df_ohlcv = pd.read_parquet(codebase_path / "output/df_ohlcv.parquet")
df_indices = pd.read_parquet(codebase_path / "output/df_indices.parquet")
df_fed = pd.read_parquet(codebase_path / "output/df_fed.parquet")
features_df = pd.read_parquet(codebase_path / "output/features_df.parquet")
macro_df = pd.read_parquet(codebase_path / "output/macro_df.parquet")
print("Data loaded successfully.")

In [ ]:
# CELL 3: Imports
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from typing import cast

# --- Domain Imports ---
from core.settings import TradingConfig
from data_pipeline.screener import UniverseScreener
from data_pipeline.cache import AlphaCache
from rl_discovery.oracle import RLOracle
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.trainer import RolloutBuffer, PPOTrainer
from rl_discovery.validator import AgentEvaluator

In [ ]:
# ============================================================================
# TASK 1: DATA PIPELINE PREP & ALPHA CACHE
# ============================================================================
print("\n--- [TASK 1] Initializing Universe and AlphaCache ---")

# Setup Config & Extract Wide Close Prices
config = TradingConfig()
# Convert MultiIndex OHLCV to Wide Format (Dates x Tickers)
df_close = df_ohlcv["Adj Close"].unstack(level=0)
# Ensure clean calendar
trading_calendar = pd.DatetimeIndex(df_close.dropna(how="all").index.sort_values())

# 1. Initialize Screener (The Gatekeeper)
screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

# 2. Pre-compute State Space (Alpha Matrix)
# We use a 189-day lookback as our standard strategy evaluation window
cache = AlphaCache(screener=screener, config=config, lookbacks=[189])

train_start = pd.Timestamp("2000-01-01")
# train_start = pd.Timestamp("2021-01-01")

cache.build(start_date=train_start)

# Target cache path resolves based on environment switch
cache_file_path = project_path / "alpha_cache_189d_2021_2023.parquet"

# Save it to disk
cache.feature_cube.to_parquet(cache_file_path)
print(f"[SAVED] AlphaCache written to {cache_file_path}")

In [ ]:
# ============================================================================
# RELOAD BLOCK (run this first if kernel was restarted)
# ============================================================================
import pandas as pd
from pathlib import Path

# 1. Re-create config
config = TradingConfig()

# 2. Re-derive the same inputs used in Task 1
df_close = df_ohlcv["Adj Close"].unstack(level=0)
trading_calendar = pd.DatetimeIndex(df_close.dropna(how="all").index.sort_values())

# 3. Re-instantiate screener (lightweight, no heavy compute)
screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

# 4. Reload the heavy feature cube from disk
cache_file_path = project_path / "alpha_cache_189d_2021_2023.parquet"
feature_cube = pd.read_parquet(cache_file_path)


# Optional: wrap it back into a cache-like object if downstream expects it
class AlphaCacheStub:
    def __init__(self, feature_cube, screener, config):
        self.feature_cube = feature_cube
        self.screener = screener
        self.config = config


cache = AlphaCacheStub(feature_cube, screener, config)
print(f"[RELOADED] screener, config, and cache from {cache_file_path}")

# ============================================================================
# TASK 2: Bootstrapping RL Oracle and Environments
# ============================================================================
print("\n--- [TASK 2] Bootstrapping RL Oracle and Environments ---")

# 1. Oracle (The Veritable Reward Truth)
holding_period = 5
oracle = RLOracle(screener, config)
oracle.precompute_reward_matrix(holding_period=holding_period)

In [ ]:
# ============================================================================
# TASK 2: RL ENVIRONMENT & ORACLE INITIALIZATION
# ============================================================================
print("\n--- [TASK 2] Bootstrapping RL Oracle and Environments ---")

# 1. Oracle (The Veritable Reward Truth)
holding_period = 5
oracle = RLOracle(screener, config)
oracle.precompute_reward_matrix(holding_period=holding_period)

# 2. Time-Split Calendars (Vertical Slicing Temporal Data)
cal_train = trading_calendar[
    # (trading_calendar >= pd.Timestamp("2021-01-01"))
    (trading_calendar >= pd.Timestamp("2001-01-01"))
    & (trading_calendar < pd.Timestamp("2023-01-01"))
]
cal_test = trading_calendar[(trading_calendar >= pd.Timestamp("2023-01-01"))]

# 3. Environments
env_train = DiscoveryEnv(
    cache.feature_cube, oracle.reward_matrix, cal_train, holding_period
)
gym_train = RLVRGymEnv(env_train, macro_df)

env_test = DiscoveryEnv(
    cache.feature_cube, oracle.reward_matrix, cal_test, holding_period
)
gym_test = RLVRGymEnv(env_test, macro_df)

In [ ]:
# ============================================================================
# TASK 3: PPO TRAINING LOOP
# ============================================================================
print("\n--- [TASK 3] Executing PPO Actor-Critic Training Loop ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Accelerator: {device}")

# Neural Nets and Trainer
agent = AbsoluteZeroAgent(obs_dim=33, action_dim=13).to(device)
trainer = PPOTrainer(agent, lr=3e-4, clip_coef=0.2, ent_coef=0.01)

num_epochs = 50
num_steps = 64  # Rollout batch size

# Pre-allocate memory to avoid memory fragmentation overhead
buffer = RolloutBuffer(num_steps=num_steps, obs_dim=33, action_dim=13, device=device)

# Track metric history to save later
training_history = []

for epoch in range(num_epochs):
    obs, _ = gym_train.reset()
    epoch_rewards = []

    # 1. Rollout Phase (Data Collection)
    for step in range(num_steps):
        # [GUARD] Trap NaNs before they hit PyTorch
        if np.isnan(obs).any():
            raise ValueError(
                "NaN detected in observation. Stopping to prevent exploding gradients."
            )

        obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)

        # Acting in the environment
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(
                obs_tensor.unsqueeze(0)
            )

        # Step Environment
        next_obs, reward, done, _, info = gym_train.step(action.cpu().numpy()[0])

        # Store experience
        buffer.add(obs, action[0], logprob[0], reward, value[0], done)
        epoch_rewards.append(reward)

        obs = next_obs
        if done:
            obs, _ = gym_train.reset()

    # 2. Advantage Calculation (GAE)
    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        next_value = agent.get_value(obs_tensor).flatten()

    buffer.compute_advantages(next_value, next_done=done)

    # 3. Surrogate Objective Optimization Phase
    trainer.update(buffer, update_epochs=4, mini_batch_size=16)

    # Buffer reset implicitly handled by tracking pointers in RolloutBuffer (step = 0)
    buffer.step = 0

    mean_rew = np.mean(epoch_rewards)
    pct_gain = (np.exp(mean_rew) - 1.0) * 100

    # Store progress metrics
    training_history.append(
        {"epoch": epoch + 1, "mean_step_reward": mean_rew, "pct_gain": pct_gain}
    )

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:03d}/{num_epochs} | Mean Step Reward: {mean_rew:.6f} ({pct_gain:.2f}%)"
        )

# Save training run progress to CSV
df_history = pd.DataFrame(training_history)
history_save_path = project_path / "ppo_training_history.csv"
df_history.to_csv(history_save_path, index=False)
print(f"[SAVED] Training history metrics written to {history_save_path}")

# Save the trained Neural Network weights (using consolidated path)
model_file_path = project_path / "absolute_zero_ppo_agent_v1.pt"
torch.save(agent.state_dict(), str(model_file_path))
print(f"[SAVED] Trained Agent weights written to {model_file_path}")

In [ ]:
# ============================================================================
# TASK 4: OUT-OF-SAMPLE VALIDATION
# ============================================================================
print("\n--- [TASK 4] Walk-Forward Deterministic Evaluation (OOS) ---")

# Switch to Deterministic execution
results = AgentEvaluator.evaluate(agent, gym_test, device)

print(f"\n[OOS METRICS]")
print(f"Total Cumulative Return: {results['total_return']*100:.2f}%")
print(f"Sharpe Ratio (Annualized): {results['sharpe_ratio']:.3f}")
print(f"Total Trading Steps: {results['steps']}")

# Save the results dictionary using pickle to allow quick plotting later without re-running evaluation
import pickle

results_save_path = project_path / "oos_evaluation_results.pkl"
with open(results_save_path, "wb") as f:
    pickle.dump(results, f)
print(f"[SAVED] Complete evaluation results dictionary written to {results_save_path}")

import plotly.io as pio

# Ensure Plotly renders correctly in Colab's iframe
pio.renderers.default = "colab"

# 1. Apply the "Year 1700" Fix (Drop the first element)
x_dates = results["dates"][1:]
y_equity = results["equity_curve"][1:]

# 2. Rebuild the Chart using corrected slicing
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_dates,
        y=y_equity,
        mode="lines",
        line=dict(color="green", width=2),
        name="RL Agent Portfolio (OOS)",
    )
)

fig.update_layout(
    title=f"Absolute Zero: Out-of-Sample Agent Performance<br><sup>OOS Sharpe: {results['sharpe_ratio']:.2f} | Returns: {results['total_return']*100:.2f}%</sup>",
    yaxis_title="Cumulative Return (Base 1.0)",
    xaxis_title="Date",
    template="plotly_white",
    hovermode="x unified",
    height=500,
)

# 3. Force Render
fig.show()

# Save the results dictionary as a CSV using the corrected sliced data
df_results = pd.DataFrame({"Date": x_dates, "Equity_Curve": y_equity})
csv_save_path = project_path / "oos_equity_curve.csv"
df_results.to_csv(csv_save_path, index=False)
print(f"[SAVED] Sliced equity curve raw data saved to {csv_save_path}")

# Save the interactive Plotly chart as an HTML file (using consolidated path)
html_save_path = project_path / "oos_performance_chart.html"
fig.write_html(str(html_save_path))
print(f"[SAVED] Interactive chart saved to {html_save_path}")